### Implementing a Simple Chatbot Using LangGraph

In this notebook, we will build a conversational chatbot using LangGraph. We will cover:
- Defining conversational state with message lists
- Using the `add_messages` reducer to append messages rather than overwrite them
- Creating a chatbot node powered by Chat Models (Groq / OpenAI)
- Compiling, invoking, and streaming responses from the graph


In [ ]:
from typing import Annotated
from typing_extensions import TypedDict
from langchain_core.messages import AnyMessage, HumanMessage, AIMessage, SystemMessage
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages


#### Reducers in LangGraph State
By default, nodes overwrite previous state values. However, in conversational graphs, we want each node's response to be **appended** to the conversation history.

The `add_messages` reducer function handles this automatically by appending new messages, handling message IDs, and supporting updating existing messages.


In [ ]:
class State(TypedDict):
    # The Annotated type hint tells LangGraph to use add_messages as the reducer for this key
    messages: Annotated[list[AnyMessage], add_messages]


In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()

os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY", "")
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY", "")


#### Initializing Chat Models
You can use either Groq (fast open-source models like `llama-3.3-70b-versatile` or `qwen-qwq-32b`) or OpenAI (`gpt-4o-mini` / `gpt-4o`).


In [ ]:
# Example with Groq
from langchain_groq import ChatGroq

llm = ChatGroq(model="llama-3.3-70b-versatile")
# Alternatively with OpenAI:
# from langchain_openai import ChatOpenAI
# llm = ChatOpenAI(model="gpt-4o-mini")

response = llm.invoke("Hey I am Krish and I like to play cricket")
print(response.content)


### Creating Graph Nodes
Our chatbot node receives the current `State` (containing the list of messages), passes `state["messages"]` to the LLM, and returns the response in a list. The `add_messages` reducer will append this response to the state.


In [ ]:
def superbot(state: State) -> dict:
    return {"messages": [llm.invoke(state["messages"])]}


### Building and Compiling the Graph


In [ ]:
from IPython.display import Image, display

builder = StateGraph(State)

## Add node
builder.add_node("SuperBot", superbot)

## Add Edges
builder.add_edge(START, "SuperBot")
builder.add_edge("SuperBot", END)

## Compile
graph = builder.compile()

## Display graph structure
try:
    display(Image(graph.get_graph().draw_mermaid_png()))
except Exception:
    print(graph.get_graph().draw_mermaid())


### Invocation
Let's invoke the graph with a user message.


In [ ]:
input_message = HumanMessage(content="Hi, My name is Krish and I like cricket")
output = graph.invoke({"messages": [input_message]})

for msg in output["messages"]:
    msg.pretty_print()


#### Streaming Responses
LangGraph supports multiple streaming modes:
- `stream_mode="updates"`: Streams state updates emitted after each node completes.
- `stream_mode="values"`: Streams the full state values at each step of execution.


In [ ]:
print("--- Streaming Updates ---")
for event in graph.stream(
    {"messages": [HumanMessage(content="Hello, can you give me 3 quick tips to improve batting in cricket?")]},
    stream_mode="updates"
):
    print(event)


In [ ]:
print("--- Streaming Values ---")
for state_snapshot in graph.stream(
    {"messages": [HumanMessage(content="Tell me one quick fact about cricket history.")]},
    stream_mode="values"
):
    print(f"Total messages in state: {len(state_snapshot['messages'])}")
